# Visualization Layer Preparation - Gold Layer

Prepares optimized visualization tables for the Streamlit app with pre-computed metrics and geometries.

**Key Optimizations:**
- Pre-computed distances to existing stores (avoids runtime calculation)
- Pre-computed partner store proximity via spatial joins (replaces point-in-polygon checks)
- Pre-computed optimization results for 27 parameter combinations (O(1) lookup vs O(n²) runtime)
- Pre-aggregated network metrics (eliminates loops)
- Broadcast cross join for nearest store calculation (modest performance improvement)

**Inputs:**
- `{catalog}.{gold_schema}.candidates_finalized` - Scored candidates with predictions
- `{catalog}.{silver_schema}.whitespace_locations` - Pre-computed distances
- `{catalog}.{silver_schema}.current_stores_features_agg` - Existing store data
- `{catalog}.{silver_schema}.pois_competitors` - Competitor locations
- `{catalog}.{silver_schema}.isochrones_partners` - Partner store trade areas
- `{catalog}.{bronze_schema}.census_states` - State boundaries

**Outputs:**
- `{catalog}.{gold_schema}.viz_h3_grid` - MA H3-8 grid cells
- `{catalog}.{gold_schema}.viz_expansion_candidates` - Enhanced candidates with distance, proximity, strategy
- `{catalog}.{gold_schema}.viz_existing_stores` - Current stores with geometries
- `{catalog}.{gold_schema}.viz_competitors` - Competitor locations
- `{catalog}.{gold_schema}.viz_partners` - Partner isochrones with candidate counts
- `{catalog}.{gold_schema}.viz_network_metrics` - Singleton aggregate KPIs
- `{catalog}.{gold_schema}.viz_optimization_results` - Pre-computed optimizations

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit, when
from pyspark.sql.window import Window

# Phase 6.1: Removed hardcoded catalog default - use bundle variables
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "geo_bronze")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("gold_schema", "geo_gold")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

# Validate required parameters
assert catalog, "ERROR: catalog parameter is required. Set via bundle variables."

print(f"Catalog: {catalog}")
print(f"Bronze: {bronze_schema}, Silver: {silver_schema}, Gold: {gold_schema}")

## 1. Generate H3 Grid Covering Massachusetts

In [ ]:
# Load Massachusetts boundary
ma_boundary = spark.table(f"{catalog}.{bronze_schema}.census_states").filter(
    (col("state_abbr") == "MA") | (col("state_fips") == "25")
)

# Generate H3-8 grid covering Massachusetts
viz_h3_grid = ma_boundary.select(
    explode(expr("h3_coverash3string(ST_AsBinary(geometry), 5)")).alias("coarse_h3")
).select(
    explode(expr("h3_tochildren(coarse_h3, 8)")).alias("h3_cell_id")
).distinct().withColumn(
    "geometry", expr("ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326)")
).withColumn(
    "center_lat", expr("ST_Y(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
).withColumn(
    "center_lon", expr("ST_X(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
)

print(f"Generated H3-8 grid with {viz_h3_grid.count()} cells")

# Write H3 grid
viz_h3_grid.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_h3_grid")
print(f"Written to {catalog}.{gold_schema}.viz_h3_grid")

## 2. Prepare Expansion Candidates

Enhanced table with:
- Normalized scores for visualization
- Pre-computed distances from whitespace_locations (converted to miles)
- Pre-computed convenience proximity via spatial joins
- Fulfillment strategy (partner vs new_store)
- Quality tier for filtering

In [ ]:
# Load expansion candidates
candidates = spark.table(f"{catalog}.{gold_schema}.candidates_finalized")
print(f"Loaded candidates: {candidates.count()}")

# Phase 3.3 OPTIMIZATION: Use pre-computed h3_cell_id from whitespace_locations
# instead of recomputing via h3_longlatash3string()
whitespace_h3 = spark.table(f"{catalog}.{silver_schema}.whitespace_locations").select(
    col("location_id").cast("string").alias("ws_location_id"),
    col("h3_cell_id").alias("ws_h3_cell_id")
)

# Join to get pre-computed h3_cell_id
candidates = candidates.join(
    whitespace_h3,
    candidates["candidate_id"].cast("string") == whitespace_h3["ws_location_id"],
    "left"
).withColumn(
    "h3_cell_id",
    F.coalesce(col("ws_h3_cell_id"), expr("h3_longlatash3string(longitude, latitude, 8)"))
).drop("ws_location_id", "ws_h3_cell_id")

print(f"Joined h3_cell_id from whitespace_locations")

# Load existing stores for nearest store ID (note: crossJoin optimization moved to cell below)
existing_stores = spark.table(f"{catalog}.{silver_schema}.current_stores_features_agg").select(
    col("store_number"),
    col("latitude").alias("store_lat"),
    col("longitude").alias("store_lon")
)
print(f"Loaded existing stores: {existing_stores.count()}")

# Load partner isochrones for proximity check
partner_isochrones = spark.table(f"{catalog}.{silver_schema}.isochrones_partners").select(
    col("location_id").alias("partner_id"),
    col("store_type").alias("partner_store_type"),
    col("city").alias("partner_city"),
    col("store_type").alias("partner_store_name"),
    col("drive_time_minutes").alias("partner_drive_time"),
    col("geometry").alias("partner_geometry")
)

print(f"Partner isochrones: {partner_isochrones.count()}")

### 2.1 Join Pre-computed Distance and Nearest Store ID

Uses pre-computed values from whitespace_locations (eliminates O(n²) crossJoin).

Phase 3.1 Optimization: The distance to nearest store and nearest_store_id are now computed
upstream in create_whitespace_locations.ipynb and reused here via a simple join.

In [ ]:
# Join candidates with pre-computed distance AND nearest_store_id from whitespace_locations
# Phase 3.1 OPTIMIZATION: Eliminates O(n²) crossJoin by using pre-computed values
# whitespace_locations already has both distance_to_nearest_current_store and nearest_store_id

# Note: whitespace_locations stores distance in miles
whitespace_with_nearest = spark.table(f"{catalog}.{silver_schema}.whitespace_locations").select(
    col("location_id").cast("string").alias("ws_location_id"),
    col("distance_to_nearest_current_store").alias("distance_to_nearest_miles"),
    col("nearest_store_id").alias("ws_nearest_store_id")
)
print(f"Loaded whitespace locations with nearest store info: {whitespace_with_nearest.count()}")

# Join candidates with pre-computed distance and nearest_store_id from whitespace_locations
candidate_distances = candidates.join(
    whitespace_with_nearest,
    candidates["candidate_id"].cast("string") == whitespace_with_nearest["ws_location_id"],
    "left"
).withColumn(
    "min_distance_to_existing",
    F.coalesce(col("distance_to_nearest_miles"), lit(999.0))  # Already in miles
).withColumn(
    "nearest_existing_store",
    col("ws_nearest_store_id")
).drop("ws_location_id", "distance_to_nearest_miles", "ws_nearest_store_id")

print(f"Joined distances and nearest store for {candidate_distances.count()} candidates")
print("\nDistance distribution (miles):")
candidate_distances.select("min_distance_to_existing").summary().show()

### 2.2 Pre-compute Partner Store Proximity

Check which candidates fall within partner store 5-min isochrones using ST_CONTAINS.

In [ ]:
# Check which candidates fall within partner isochrones using ST_CONTAINS
candidates_with_point = candidates.withColumn(
    "candidate_point", expr("ST_SetSRID(ST_Point(longitude, latitude), 4326)")
)

# Spatial join: candidate point within isochrone polygon
candidates_in_partners = candidates_with_point.join(
    partner_isochrones,
    expr("ST_Contains(partner_geometry, candidate_point)"),
    "left"
)

# Keep shortest drive time for candidates in multiple isochrones
window_partner = Window.partitionBy("candidate_id").orderBy("partner_drive_time")

partner_proximity = candidates_in_partners.withColumn(
    "row_num", F.row_number().over(window_partner)
).filter(col("row_num") == 1).select(
    "candidate_id",
    when(col("partner_id").isNotNull(), True).otherwise(False).alias("within_partner_isochrone"),
    col("partner_id"),
    col("partner_store_type"),
    col("partner_city"),
    col("partner_store_name"),
    col("partner_drive_time")
)

in_isochrone_count = partner_proximity.filter(col("within_partner_isochrone") == True).count()
print(f"Candidates within partner isochrones: {in_isochrone_count}")
print(f"Candidates outside: {partner_proximity.count() - in_isochrone_count}")

### 2.3 Assemble Enhanced Expansion Candidates Table

Add normalized scores, quality tier, fulfillment strategy, and geometries.

In [ ]:
# Calculate min/max for normalization
stats = candidates.agg(
    F.min("predicted_annual_sales").alias("min_sales"),
    F.max("predicted_annual_sales").alias("max_sales"),
    F.min("population").alias("min_pop"),
    F.max("population").alias("max_pop")
).collect()[0]

min_sales, max_sales = stats["min_sales"], stats["max_sales"]
min_pop, max_pop = stats["min_pop"], stats["max_pop"]

# Join candidates with distance data (from candidate_distances) and partner proximity
# Note: Join on candidate_id (store-level identifier)
viz_candidates = candidate_distances.join(
    partner_proximity, "candidate_id", "left"
)

# Check if urbanity_category exists, otherwise use urbanity or default
available_cols = viz_candidates.columns
if "urbanity_category" in available_cols:
    urbanity_col = col("urbanity_category")
elif "urbanity" in available_cols:
    # Map urbanity to category
    urbanity_col = when(col("urbanity").isin("Very_High_density_urban", "High_density_urban"), "urban") \
                   .when(col("urbanity").isin("Medium_density_urban", "Low_density_urban"), "suburban") \
                   .otherwise("rural")
else:
    urbanity_col = lit("unknown")

# Add city column: use partner_city if available, otherwise derive from urbanity
viz_candidates = viz_candidates.withColumn(
    "city",
    when(col("partner_city").isNotNull(), col("partner_city"))
    .otherwise(
        when(urbanity_col == "urban", "Boston Metro")
        .when(urbanity_col == "suburban", "Greater Boston")
        .otherwise("Massachusetts")
    )
)

# Add normalized scores
if max_sales > min_sales:
    viz_candidates = viz_candidates.withColumn(
        "normalized_sales_score",
        (col("predicted_annual_sales") - lit(min_sales)) / lit(max_sales - min_sales)
    )
else:
    viz_candidates = viz_candidates.withColumn("normalized_sales_score", lit(0.5))

if max_pop > min_pop:
    viz_candidates = viz_candidates.withColumn(
        "normalized_pop_score",
        (col("population") - lit(min_pop)) / lit(max_pop - min_pop)
    )
else:
    viz_candidates = viz_candidates.withColumn("normalized_pop_score", lit(0.5))

# Add percentile rank, quality tier, fulfillment strategy, and geometry columns
# Note: h3_cell_id is generated from lat/lon, used for H3 geometry functions
viz_candidates = viz_candidates.withColumn(
    "percentile_rank",
    F.percent_rank().over(Window.orderBy("predicted_annual_sales"))
).withColumn(
    "quality_tier",
    when(col("percentile_rank") >= 0.75, "top_25")
    .when(col("percentile_rank") >= 0.50, "top_50")
    .when(col("percentile_rank") >= 0.25, "top_75")
    .otherwise("bottom_25")
).withColumn(
    "fulfillment_strategy",
    when(col("within_partner_isochrone") == True, "partner").otherwise("new_store")
).withColumn(
    "geometry", expr("ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326)")
).withColumn(
    "geometry_geojson", expr("ST_AsGeoJSON(ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326))")
).withColumn(
    "center_lat", expr("ST_Y(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
).withColumn(
    "center_lon", expr("ST_X(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
)

# Fill nulls for distance and partner columns
viz_candidates = viz_candidates.fillna({
    "min_distance_to_existing": 999.0,
    "within_partner_isochrone": False
})

print(f"Enhanced candidates with {viz_candidates.count()} rows")
print("\nColumn list:")
print(viz_candidates.columns)

# Write enhanced table
viz_candidates.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_expansion_candidates")
print(f"\nWritten to {catalog}.{gold_schema}.viz_expansion_candidates")

### 2.4 AI Feasibility Score (Optional)

Use AI_QUERY() to evaluate location feasibility based on factors not captured in features (proximity to water, highways, visibility). Currently disabled.

In [ ]:
# AI Feasibility Score - COMMENTED OUT FOR NOW
# This cell uses ai_query() to evaluate location feasibility based on factors NOT captured in features.
# Uncomment when ready to enable AI-powered feasibility scoring.

# try:
#     print("Computing AI Feasibility Scores for new_store candidates...")
#     
#     # Filter to only new_store candidates (partner candidates don't need this)
#     new_store_candidates = viz_candidates.filter(col("fulfillment_strategy") == "new_store")
#     partner_candidates = viz_candidates.filter(col("fulfillment_strategy") == "partner")
#     
#     print(f"  New store candidates: {new_store_candidates.count()}")
#     print(f"  Partner candidates: {partner_candidates.count()}")
#     
#     # Register as temp view for SQL ai_query
#     new_store_candidates.createOrReplaceTempView("new_store_candidates_temp")
#     
#     # Use ai_query to evaluate feasibility for each candidate
#     ai_evaluated = spark.sql("""
#         SELECT 
#             candidate_id,
#             latitude,
#             longitude,
#             ai_query(
#                 'databricks-gpt-5-2',
#                 CONCAT(
#                     'You are a retail site selection expert for Little Caesars Pizza. ',
#                     'Evaluate the feasibility of opening a new restaurant at coordinates: ',
#                     CAST(latitude AS STRING), ', ', CAST(longitude AS STRING), ' in Massachusetts. ',
#                     'Consider ONLY these location-specific factors NOT captured in demographic data: ',
#                     '1) Proximity to bodies of water, parks, or green spaces that limit development ',
#                     '2) Proximity to major highways and accessibility ',
#                     '3) Visibility from main roads ',
#                     '4) Physical barriers (railroads, rivers, hills) affecting access ',
#                     '5) Nearby landmarks or attractions affecting foot traffic. ',
#                     'Provide a feasibility score from 1-5 (5=highest feasibility) and a brief 2-sentence rationale. ',
#                     'Format your response EXACTLY as: SCORE: [1-5] | RATIONALE: [your 2-sentence explanation]'
#                 )
#             ) as ai_response
#         FROM new_store_candidates_temp
#     """)
#     
#     # Parse the AI response to extract score and rationale
#     ai_parsed = ai_evaluated.withColumn(
#         "ai_feasibility_score",
#         F.regexp_extract(col("ai_response"), r"SCORE:\s*(\d)", 1).cast("int")
#     ).withColumn(
#         "ai_feasibility_rationale",
#         F.regexp_extract(col("ai_response"), r"RATIONALE:\s*(.+)", 1)
#     ).drop("ai_response")
#     
#     # Handle parsing failures - default to score 3 (neutral) if parsing fails
#     ai_parsed = ai_parsed.withColumn(
#         "ai_feasibility_score",
#         when(col("ai_feasibility_score").isNull(), 3).otherwise(col("ai_feasibility_score"))
#     ).withColumn(
#         "ai_feasibility_rationale",
#         when(col("ai_feasibility_rationale").isNull(), "AI evaluation unavailable").otherwise(col("ai_feasibility_rationale"))
#     )
#     
#     print(f"  AI evaluation complete for {ai_parsed.count()} candidates")
#     
#     # Join AI scores back to new_store candidates using candidate_id
#     new_store_with_ai = new_store_candidates.join(
#         ai_parsed.select("candidate_id", "ai_feasibility_score", "ai_feasibility_rationale"),
#         "candidate_id",
#         "left"
#     )
#     
#     # For partner candidates, set AI scores to null (they don't need this)
#     partner_with_nulls = partner_candidates.withColumn(
#         "ai_feasibility_score", lit(None).cast("int")
#     ).withColumn(
#         "ai_feasibility_rationale", lit(None).cast("string")
#     )
#     
#     # Union back together
#     viz_candidates = new_store_with_ai.unionByName(partner_with_nulls)
#     
#     print(f"\n AI Feasibility Scores added to viz_candidates")
#     
#     # CRITICAL: Overwrite the table with AI-enhanced candidates
#     viz_candidates.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_expansion_candidates")
#     print(f" Re-written viz_expansion_candidates with AI scores")
#     
# except Exception as e:
#     print(f"WARNING: Could not compute AI feasibility scores: {e}")

print("AI Feasibility Score computation is currently disabled.")

## 3. Prepare Existing Stores

In [ ]:
try:
    existing_stores = spark.table(f"{catalog}.{silver_schema}.current_stores_features_agg")
    
    # Get column list to handle optional columns
    existing_cols = existing_stores.columns
    
    print(f"Available columns in current_stores_features_agg: {existing_cols}")
    
    # Calculate total POI count by summing individual category columns
    # Note: agg_h3_features_current_stores creates columns named: retail, food_drink, leisure, etc.
    # NOT total_retail_pois, total_food_drink_pois, etc.
    poi_categories = ['retail', 'food_drink', 'leisure', 'education', 'healthcare', 'financial', 'tourism', 'transportation']
    
    # Build sum expression for available POI columns
    poi_sum_expr = lit(0)
    for poi_cat in poi_categories:
        if poi_cat in existing_cols:
            poi_sum_expr = poi_sum_expr + F.coalesce(col(poi_cat), lit(0))
    
    # If total_poi_count already exists, use it; otherwise calculate it
    if 'total_poi_count' in existing_cols:
        existing_with_poi = existing_stores.withColumn("poi_count_calc", col("total_poi_count"))
    else:
        existing_with_poi = existing_stores.withColumn("poi_count_calc", poi_sum_expr)
    
    viz_existing = existing_with_poi.select(
        "store_number",
        "latitude",
        "longitude",
        col("store_type") if "store_type" in existing_cols else lit("LCE").alias("store_type"),
        "city",
        "state",
        "population",
        "annual_sales",
        col("poi_count_calc").alias("poi_count"),
        "geometry"
    ).withColumn(
        "marker_type", lit("existing_lce")
    ).withColumn(
        # Add H3 cell ID for consistent filtering with viz_h3_grid
        # Note: h3_longlatash3string takes (longitude, latitude, resolution)
        "h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)")
    ).withColumn(
        "geometry_geojson", expr("ST_AsGeoJSON(geometry)")
    )
    
    print(f"Prepared {viz_existing.count()} existing stores")
    
    viz_existing.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_existing_stores")
    print(f"✓ Written to {catalog}.{gold_schema}.viz_existing_stores")
    
except Exception as e:
    print(f"✗ ERROR preparing existing stores: {e}")
    import traceback
    traceback.print_exc()

## 4. Prepare Competitors

In [ ]:
try:
    competitors = spark.table(f"{catalog}.{silver_schema}.pois_competitors")
    
    viz_competitors = competitors.select(
        col("poi_id").alias("id"),
        "name",
        "latitude",
        "longitude",
        "poi_category",
        "poi_subcategory",
        "address"
    ).withColumn(
        "marker_type", lit("competitor")
    ).withColumn(
        "geometry", expr("ST_Point(longitude, latitude)")
    ).withColumn(
        # Add H3 cell ID for consistent filtering with viz_h3_grid
        # Note: h3_longlatash3string takes (longitude, latitude, resolution)
        "h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)")
    ).withColumn(
        "geometry_geojson", expr("ST_AsGeoJSON(ST_Point(longitude, latitude))")
    )
    
    print(f"Prepared {viz_competitors.count()} competitors")
    
    viz_competitors.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_competitors")
    print(f"✓ Written to {catalog}.{gold_schema}.viz_competitors")
    
except Exception as e:
    print(f"✗ ERROR preparing competitors: {e}")
    import traceback
    traceback.print_exc()

## 5. Prepare Partner Store Isochrones (Enhanced with Candidate Proximity)

This table now includes:
- Count of expansion candidates within each partner isochrone
- Total predicted sales of candidates in isochrone (partnership potential)
- Array of candidate H3 cell IDs within isochrone for reverse lookup

In [ ]:
try:
    partners = spark.table(f"{catalog}.{silver_schema}.isochrones_partners")
    candidates_for_partners = spark.table(f"{catalog}.{gold_schema}.candidates_finalized")

    # Check available columns in partners table for poi_subcategory
    partner_cols = partners.columns
    has_poi_subcategory = "poi_subcategory" in partner_cols
    print(f"Partner table columns: {partner_cols}")
    print(f"Has poi_subcategory: {has_poi_subcategory}")

    # Generate h3_cell_id from lat/lon (candidate_id is store-level, h3_cell_id is hexagon grid)
    candidates_for_partners = candidates_for_partners.withColumn(
        "h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)")
    )

    # Create point geometry for candidates with SRID 4326 to match isochrone geometry
    candidates_with_point = candidates_for_partners.withColumn(
        "candidate_point", expr("ST_SetSRID(ST_Point(longitude, latitude), 4326)")
    ).select("candidate_id", "h3_cell_id", "candidate_point", "predicted_annual_sales")
    
    # Join partner isochrones with candidates that fall within them
    partners_with_candidates = partners.alias("p").join(
        candidates_with_point.alias("cand"),
        expr("ST_Contains(p.geometry, cand.candidate_point)"),
        "left"
    )
    
    # Aggregate candidate info per partner store
    # Use candidate_id for counting, h3_cell_id for the array (for downstream H3 lookups)
    candidate_agg = partners_with_candidates.groupBy(
        col("p.location_id")
    ).agg(
        F.count("cand.candidate_id").alias("candidate_count_in_isochrone"),
        F.sum("cand.predicted_annual_sales").alias("total_candidate_sales_in_isochrone"),
        F.collect_list("cand.h3_cell_id").alias("candidate_h3_cells_in_isochrone")
    )
    
    # Build select list with optional poi_subcategory (Phase 5.1)
    select_cols = [
        col("part.location_id").alias("id"),
        col("part.store_type"),
        col("part.store_type").alias("name"),  # Use store_type as name (e.g., "Walmart", "7-Eleven")
        col("part.latitude"),
        col("part.longitude"),
        col("part.city"),
        col("part.state"),
        col("part.drive_time_minutes"),
        col("part.area_sqkm"),
        col("part.geometry"),
        F.coalesce(col("agg.candidate_count_in_isochrone"), lit(0)).alias("candidate_count_in_isochrone"),
        F.coalesce(col("agg.total_candidate_sales_in_isochrone"), lit(0)).alias("total_candidate_sales_in_isochrone"),
        F.coalesce(col("agg.candidate_h3_cells_in_isochrone"), F.array()).alias("candidate_h3_cells_in_isochrone")
    ]
    
    # Phase 5.1: Add poi_subcategory if available in source table
    if has_poi_subcategory:
        select_cols.append(col("part.poi_subcategory"))
    
    # Join back to partners base table (use aliases to avoid ambiguity)
    viz_partners = partners.alias("part").join(
        candidate_agg.alias("agg"),
        col("part.location_id") == col("agg.location_id"),
        "left"
    ).select(*select_cols).withColumn(
        "marker_type", lit("partner")
    ).withColumn(
        "geometry_geojson", expr("ST_AsGeoJSON(geometry)")
    ).withColumn(
        # Note: h3_longlatash3string takes (longitude, latitude, resolution)
        "h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)")
    )
    
    # Add poi_subcategory as null if not in source table (for consistent schema)
    if not has_poi_subcategory:
        viz_partners = viz_partners.withColumn("poi_subcategory", lit(None).cast("string"))
    
    print(f"Prepared {viz_partners.count()} partner store isochrones")
    
    # Show partnership potential summary
    print("\nPartnership potential summary:")
    viz_partners.select(
        "candidate_count_in_isochrone", "total_candidate_sales_in_isochrone"
    ).summary().show()
    
    # Show partner breakdown by store type
    print("\nPartner stores by type:")
    viz_partners.groupBy("store_type").count().orderBy(F.desc("count")).show()
    
    viz_partners.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_partners")
    print(f"Written to {catalog}.{gold_schema}.viz_partners")
    
except Exception as e:
    print(f"ERROR preparing partner isochrones: {e}")
    import traceback
    traceback.print_exc()

## 6. Create Network Metrics (Singleton Aggregates)

Pre-computed aggregate metrics for dashboard display. This eliminates runtime aggregation loops in the app.

In [ ]:
from datetime import datetime

try:
    # Load data for metrics calculation
    existing = spark.table(f"{catalog}.{silver_schema}.current_stores_features_agg")
    candidates = spark.table(f"{catalog}.{gold_schema}.candidates_finalized")
    
    # Get column list to handle optional columns
    existing_cols = existing.columns
    
    # Calculate total POI count by summing individual category columns
    # Note: agg_h3_features_current_stores creates columns named: retail, food_drink, leisure, etc.
    poi_categories = ['retail', 'food_drink', 'leisure', 'education', 'healthcare', 'financial', 'tourism', 'transportation']
    
    # Build sum expression for available POI columns
    poi_sum_expr = lit(0)
    for poi_cat in poi_categories:
        if poi_cat in existing_cols:
            poi_sum_expr = poi_sum_expr + F.coalesce(col(poi_cat), lit(0))
    
    # If total_poi_count already exists, use it; otherwise calculate it
    if 'total_poi_count' in existing_cols:
        existing_with_poi = existing.withColumn("total_poi_calc", col("total_poi_count"))
    else:
        existing_with_poi = existing.withColumn("total_poi_calc", poi_sum_expr)
    
    # Calculate existing store metrics
    existing_metrics = existing_with_poi.agg(
        F.count("*").alias("total_existing_stores"),
        F.avg("population").alias("avg_store_population"),
        F.avg("total_poi_calc").alias("avg_store_poi"),
        F.sum("population").alias("total_network_population")
    ).collect()[0]
    
    # Calculate candidate metrics
    candidate_metrics = candidates.agg(
        F.count("*").alias("total_candidates"),
        F.avg("predicted_annual_sales").alias("avg_candidate_sales"),
        F.avg("population").alias("avg_candidate_population"),
        F.percentile_approx("predicted_annual_sales", 0.25).alias("sales_p25"),
        F.percentile_approx("predicted_annual_sales", 0.50).alias("sales_p50"),
        F.percentile_approx("predicted_annual_sales", 0.75).alias("sales_p75"),
        F.min("predicted_annual_sales").alias("sales_min"),
        F.max("predicted_annual_sales").alias("sales_max")
    ).collect()[0]
    
    # Create singleton metrics row (use Python datetime instead of F.current_timestamp())
    viz_network_metrics = spark.createDataFrame([{
        "total_existing_stores": int(existing_metrics["total_existing_stores"]),
        "avg_store_population": float(existing_metrics["avg_store_population"] or 0),
        "avg_store_poi": float(existing_metrics["avg_store_poi"] or 0),
        "total_network_population": int(existing_metrics["total_network_population"] or 0),
        "total_candidates": int(candidate_metrics["total_candidates"]),
        "avg_candidate_sales": float(candidate_metrics["avg_candidate_sales"] or 0),
        "avg_candidate_population": float(candidate_metrics["avg_candidate_population"] or 0),
        "sales_p25": float(candidate_metrics["sales_p25"] or 0),
        "sales_p50": float(candidate_metrics["sales_p50"] or 0),
        "sales_p75": float(candidate_metrics["sales_p75"] or 0),
        "sales_min": float(candidate_metrics["sales_min"] or 0),
        "sales_max": float(candidate_metrics["sales_max"] or 0),
        "last_updated": datetime.now()
    }])
    
    print("Network Metrics Summary:")
    print(f"  Existing stores: {existing_metrics['total_existing_stores']}")
    print(f"  Avg store population: {existing_metrics['avg_store_population']:,.0f}")
    print(f"  Total candidates: {candidate_metrics['total_candidates']}")
    print(f"  Avg candidate sales: ${candidate_metrics['avg_candidate_sales']:,.0f}")
    print(f"  Sales percentiles (25/50/75): ${candidate_metrics['sales_p25']:,.0f} / ${candidate_metrics['sales_p50']:,.0f} / ${candidate_metrics['sales_p75']:,.0f}")
    
    viz_network_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_network_metrics")
    print(f"\n✓ Written to {catalog}.{gold_schema}.viz_network_metrics")
    
except Exception as e:
    print(f"✗ ERROR creating network metrics: {e}")
    import traceback
    traceback.print_exc()

## 7. Pre-compute Optimization Results

Pre-compute optimization for 27 parameter combinations (3×3×3 grid):
- max_stores: [10, 50, 100]
- min_distance_new: [1.0, 2.0, 3.0] miles
- min_distance_existing: [1.0, 2.0, 3.0] miles

Reduces latency from 500-2000ms to <50ms.

In [ ]:
from itertools import product
import math

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in miles between two points using Haversine formula."""
    R = 3959  # Earth's radius in miles
    
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    
    return R * c

def run_greedy_optimization(candidates_pdf, existing_stores_pdf, max_stores, min_dist_new, min_dist_existing):
    """
    Greedy optimization algorithm that selects candidates based on:
    1. Highest predicted sales first
    2. Must be >= min_dist_existing from all existing stores
    3. Must be >= min_dist_new from all already-selected candidates
    """
    # Sort by predicted sales descending
    sorted_candidates = candidates_pdf.sort_values('predicted_annual_sales', ascending=False)
    
    selected = []
    selected_coords = []
    
    for _, candidate in sorted_candidates.iterrows():
        if len(selected) >= max_stores:
            break
            
        cand_lat = candidate['latitude']
        cand_lon = candidate['longitude']
        
        # Check distance to existing stores
        too_close_to_existing = False
        for _, store in existing_stores_pdf.iterrows():
            dist = haversine_distance(cand_lat, cand_lon, store['latitude'], store['longitude'])
            if dist < min_dist_existing:
                too_close_to_existing = True
                break
        
        if too_close_to_existing:
            continue
            
        # Check distance to already-selected candidates
        too_close_to_selected = False
        for sel_lat, sel_lon in selected_coords:
            dist = haversine_distance(cand_lat, cand_lon, sel_lat, sel_lon)
            if dist < min_dist_new:
                too_close_to_selected = True
                break
        
        if too_close_to_selected:
            continue
            
        # Candidate passes all checks
        selected.append(candidate['h3_cell_id'])
        selected_coords.append((cand_lat, cand_lon))
    
    return selected

# Parameter grid - REDUCED from 96 to 27 combinations (72% reduction)
max_stores_options = [10, 50, 100]           # Small, Medium, Large expansion
min_dist_new_options = [1.0, 2.0, 3.0]       # Close, Medium, Far (miles)
min_dist_existing_options = [1.0, 2.0, 3.0]  # Close, Medium, Far (miles)

print(f"Parameter grid: {len(max_stores_options)} × {len(min_dist_new_options)} × {len(min_dist_existing_options)} = {len(max_stores_options) * len(min_dist_new_options) * len(min_dist_existing_options)} combinations")

# Load data as pandas for efficient iteration
candidates_pdf = spark.table(f"{catalog}.{gold_schema}.viz_expansion_candidates").select(
    "h3_cell_id", "latitude", "longitude", "predicted_annual_sales"
).toPandas()

existing_stores_pdf = spark.table(f"{catalog}.{silver_schema}.current_stores_features_agg").select(
    "store_number", "latitude", "longitude"
).toPandas()

print(f"Loaded {len(candidates_pdf)} candidates and {len(existing_stores_pdf)} existing stores")

In [ ]:
# Run optimization for all parameter combinations
from datetime import datetime

results = []
total_combinations = len(max_stores_options) * len(min_dist_new_options) * len(min_dist_existing_options)
current = 0

print("Running optimization for all parameter combinations...")

for max_stores, min_dist_new, min_dist_existing in product(max_stores_options, min_dist_new_options, min_dist_existing_options):
    current += 1
    
    # Run greedy optimization
    selected_h3_cells = run_greedy_optimization(
        candidates_pdf, existing_stores_pdf, 
        max_stores, min_dist_new, min_dist_existing
    )
    
    # Calculate total predicted sales for selected candidates
    selected_sales = candidates_pdf[candidates_pdf['h3_cell_id'].isin(selected_h3_cells)]['predicted_annual_sales'].sum()
    
    results.append({
        'max_stores': max_stores,
        'min_distance_new': min_dist_new,
        'min_distance_existing': min_dist_existing,
        'selected_h3_cells': selected_h3_cells,
        'selected_count': len(selected_h3_cells),
        'total_predicted_sales': float(selected_sales),
        'computed_at': datetime.now()
    })
    
    if current % 20 == 0:
        print(f"  Progress: {current}/{total_combinations} ({100*current/total_combinations:.0f}%)")

print(f"\nCompleted {len(results)} optimization results")

# Show sample results
print("\nSample results:")
for r in results[:5]:
    print(f"  max={r['max_stores']}, dist_new={r['min_distance_new']}, dist_exist={r['min_distance_existing']} -> {r['selected_count']} stores, ${r['total_predicted_sales']:,.0f}")

In [ ]:
# Convert results to Spark DataFrame and write to Delta
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, ArrayType, StringType, TimestampType

schema = StructType([
    StructField("max_stores", IntegerType(), False),
    StructField("min_distance_new", DoubleType(), False),
    StructField("min_distance_existing", DoubleType(), False),
    StructField("selected_h3_cells", ArrayType(StringType()), False),
    StructField("selected_count", IntegerType(), False),
    StructField("total_predicted_sales", DoubleType(), False),
    StructField("computed_at", TimestampType(), False)
])

optimization_results_df = spark.createDataFrame(results, schema)

print(f"Writing {optimization_results_df.count()} optimization results to Delta...")

optimization_results_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_optimization_results")

print(f"Written to {catalog}.{gold_schema}.viz_optimization_results")

# Show statistics
print("\nOptimization results statistics:")
optimization_results_df.select("selected_count", "total_predicted_sales").summary().show()

## Summary

In [ ]:
print("=" * 60)
print("VISUALIZATION LAYER SUMMARY")
print("=" * 60)

viz_tables = [
    "viz_h3_grid",
    "viz_expansion_candidates",
    "viz_existing_stores",
    "viz_competitors",
    "viz_partners",
    "viz_network_metrics",
    "viz_optimization_results"
]

for table_name in viz_tables:
    try:
        count = spark.table(f"{catalog}.{gold_schema}.{table_name}").count()
        print(f"  {table_name}: {count:,} rows")
    except Exception as e:
        print(f"  {table_name}: NOT AVAILABLE - {e}")

print("\n" + "=" * 60)
print("VISUALIZATION LAYER COMPLETE")
print("=" * 60)

# Show enhanced columns in viz_expansion_candidates
print("\nEnhanced viz_expansion_candidates columns:")
enhanced_cols = ["min_distance_to_existing", "nearest_existing_store", 
                 "within_partner_isochrone", "partner_store_name",
                 "fulfillment_strategy", "quality_tier", "geometry_geojson"]
for col_name in enhanced_cols:
    print(f"  ✓ {col_name}")

## Folium Map Visualization

Interactive map showing:
- **LCE Stores** (green markers)
- **LCE Trade Areas** (green isochrones)
- **Convenience Store Trade Areas** (blue isochrones)
- **Expansion Candidates** (red markers)

In [ ]:
try:
    %pip install folium --quiet

    import folium
    from folium.plugins import Fullscreen
    import pandas as pd
    from shapely import wkt
    from shapely.geometry import mapping

    print("Loading data for Folium map...")

    # Load LCE stores
    lce_stores_pd = (
        spark.table(f"{catalog}.{gold_schema}.viz_existing_stores")
        .select("store_number", "latitude", "longitude", "city", "state", "population")
        .toPandas()
    )

    # Load LCE isochrones as GeoJSON
    lce_isochrones_df = spark.table(f"{catalog}.{silver_schema}.isochrones_lce")
    lce_isochrones_gdf = (
        lce_isochrones_df
        .selectExpr(
            "location_id as store_number",
            "ST_AsText(geometry) as geometry_wkt",
            "drive_time_minutes",
            "area_sqkm"
        )
        .toPandas()
    )

    lce_isochrones_geojson = {
        "type": "FeatureCollection",
        "features": []
    }

    for _, row in lce_isochrones_gdf.iterrows():
        geom = wkt.loads(row['geometry_wkt'])
        feature = {
            "type": "Feature",
            "properties": {
                "store_number": row['store_number'],
                "drive_time_minutes": row['drive_time_minutes'],
                "area_sqkm": row['area_sqkm']
            },
            "geometry": mapping(geom)
        }
        lce_isochrones_geojson["features"].append(feature)

    # Load partner store isochrones as GeoJSON
    partner_isochrones_df = spark.table(f"{catalog}.{silver_schema}.isochrones_partners")
    partner_isochrones_gdf = (
        partner_isochrones_df
        .selectExpr(
            "location_id",
            "store_type",
            "ST_AsText(geometry) as geometry_wkt",
            "drive_time_minutes",
            "area_sqkm"
        )
        .toPandas()
    )

    partner_isochrones_geojson = {
        "type": "FeatureCollection",
        "features": []
    }

    for _, row in partner_isochrones_gdf.iterrows():
        geom = wkt.loads(row['geometry_wkt'])
        feature = {
            "type": "Feature",
            "properties": {
                "location_id": row['location_id'],
                "store_type": row['store_type'],
                "drive_time_minutes": row['drive_time_minutes'],
                "area_sqkm": row['area_sqkm']
            },
            "geometry": mapping(geom)
        }
        partner_isochrones_geojson["features"].append(feature)

    # Load expansion candidates (top 50 by predicted sales)
    candidates_pd = (
        spark.table(f"{catalog}.{gold_schema}.viz_expansion_candidates")
        .orderBy(F.desc("predicted_annual_sales"))
        .limit(50)
        .select("h3_cell_id", "latitude", "longitude", "predicted_annual_sales", "population")
        .toPandas()
    )

    print(f"\nLoaded data:")
    print(f"  LCE stores: {len(lce_stores_pd)}")
    print(f"  LCE trade areas: {len(lce_isochrones_geojson['features'])}")
    print(f"  Partner trade areas: {len(partner_isochrones_geojson['features'])}")
    print(f"  Expansion candidates: {len(candidates_pd)}")

    # Create Folium map centered on Massachusetts
    ma_center = [42.4072, -71.3824]
    folium_map = folium.Map(
        location=ma_center,
        zoom_start=9,
        tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
        attr='CartoDB'
    )

    print("Building map layers...")

    # Layer 1: LCE Trade Areas (GREEN isochrones)
    if lce_isochrones_geojson['features']:
        for feature in lce_isochrones_geojson['features']:
            coords = feature['geometry']['coordinates'][0]
            polygon_coords = [[lat, lon] for lon, lat in coords]

            folium.Polygon(
                locations=polygon_coords,
                color='#10b981',  # Green stroke
                fill=True,
                fillColor='#10b981',
                fillOpacity=0.1,
                weight=1,
                popup=f"<b>LCE Store {feature['properties']['store_number']}</b><br>"
                      f"Drive time: {feature['properties']['drive_time_minutes']} min<br>"
                      f"Area: {feature['properties']['area_sqkm']:.2f} km²"
            ).add_to(folium_map)

        print(f"  Added {len(lce_isochrones_geojson['features'])} LCE trade areas")

    # Layer 2: Partner Trade Areas (blue isochrones)
    if partner_isochrones_geojson['features']:
        for feature in partner_isochrones_geojson['features']:
            coords = feature['geometry']['coordinates'][0]
            polygon_coords = [[lat, lon] for lon, lat in coords]

            folium.Polygon(
                locations=polygon_coords,
                color='#3b82f6',  # Blue stroke
                fill=True,
                fillColor='#3b82f6',
                fillOpacity=0.08,
                weight=1,
                popup=f"<b>{feature['properties']['store_type']}</b><br>"
                      f"Location: {feature['properties']['location_id']}<br>"
                      f"Drive time: {feature['properties']['drive_time_minutes']} min<br>"
                      f"Area: {feature['properties']['area_sqkm']:.2f} km²"
            ).add_to(folium_map)

        print(f"  Added {len(partner_isochrones_geojson['features'])} partner trade areas")

    # Layer 3: LCE Store Markers (GREEN)
    for _, store in lce_stores_pd.iterrows():
        folium.CircleMarker(
            location=[store['latitude'], store['longitude']],
            radius=6,
            popup=f"<b>Little Caesars</b><br>"
                  f"Store: {store['store_number']}<br>"
                  f"Location: {store.get('city', 'N/A')}, {store.get('state', 'N/A')}<br>"
                  f"Population: {store.get('population', 0):,.0f}",
            tooltip=f"LCE Store {store['store_number']}",
            color='#10b981',
            fill=True,
            fillColor='#34d399',
            fillOpacity=0.8,
            weight=2
        ).add_to(folium_map)

    print(f"  Added {len(lce_stores_pd)} LCE store markers")

    # Layer 4: Expansion Candidates (RED markers)
    for _, candidate in candidates_pd.iterrows():
        folium.CircleMarker(
            location=[candidate['latitude'], candidate['longitude']],
            radius=5,
            popup=f"<b>Expansion Candidate</b><br>"
                  f"H3 Cell: {candidate['h3_cell_id']}<br>"
                  f"Predicted Sales: ${candidate['predicted_annual_sales']:,.0f}<br>"
                  f"Population: {candidate['population']:,.0f}",
            tooltip=f"Expansion: ${candidate['predicted_annual_sales']:,.0f}",
            color='#ef4444',
            fill=True,
            fillColor='#f87171',
            fillOpacity=0.7,
            weight=2
        ).add_to(folium_map)

    print(f"  Added {len(candidates_pd)} expansion candidate markers")
    print("\nMap layers complete!")

    # Add compact legend
    legend_html = '''
    <div style="position: fixed;
                bottom: 30px; right: 30px; width: 200px;
                background-color: white; border:2px solid grey; z-index:9999;
                font-size:12px; padding: 10px; border-radius: 5px; box-shadow: 0 2px 6px rgba(0,0,0,0.3);">
    <p style="margin: 0 0 8px 0; font-weight: bold; font-size: 13px;">Map Legend</p>
    <p style="margin: 5px 0;">
        <span style="display: inline-block; width: 15px; height: 15px;
                     background-color: #10b981; border-radius: 50%; border: 2px solid #059669;
                     vertical-align: middle; margin-right: 5px;"></span>
        LCE Stores
    </p>
    <p style="margin: 5px 0;">
        <span style="display: inline-block; width: 25px; height: 8px;
                     background-color: rgba(16,185,129,0.1);
                     border: 1px solid #10b981;
                     vertical-align: middle; margin-right: 5px;"></span>
        LCE Trade Areas
    </p>
    <p style="margin: 5px 0;">
        <span style="display: inline-block; width: 25px; height: 8px;
                     background-color: rgba(59,130,246,0.08);
                     border: 1px solid #3b82f6;
                     vertical-align: middle; margin-right: 5px;"></span>
        Partner Areas
    </p>
    <p style="margin: 5px 0;">
        <span style="display: inline-block; width: 15px; height: 15px;
                     background-color: #f87171; border-radius: 50%; border: 2px solid #ef4444;
                     vertical-align: middle; margin-right: 5px;"></span>
        Expansion Candidates
    </p>
    </div>
    '''

    folium_map.get_root().html.add_child(folium.Element(legend_html))

    # Add fullscreen button
    Fullscreen(
        position='topleft',
        title='Expand map',
        title_cancel='Exit fullscreen',
        force_separate_button=True
    ).add_to(folium_map)

    # Display the map
    print("\nDisplaying interactive map...")
    display(folium_map)

except Exception as e:
    print(f"Could not create Folium map: {e}")
    print("Skipping Folium visualization (this is optional)")
    import traceback
    traceback.print_exc()